# 🎬 LTX-Video Generation Pipeline via stable-diffusion.cpp

This notebook provides an optimized, end-to-end production pipeline for running the 22-Billion parameter **LTX-Video** model on Kaggle's shared system environment. By utilizing a pre-compiled `stable-diffusion.cpp` binary and highly efficient GGUF/Importance-Matrix model weights, generation times are heavily reduced while staying safely within the 15 GB VRAM hardware ceiling of a single Tesla T4 GPU.

In [ ]:
# Create the isolated, structured directory framework within the fast temporary storage layer
import os

print("📁 Creating structured workspace directories inside /tmp...")
os.makedirs("/tmp/sd_bin", exist_ok=True)
os.makedirs("/tmp/models/diffusion_models", exist_ok=True)
os.makedirs("/tmp/models/text_encoders", exist_ok=True)
os.makedirs("/tmp/models/vae", exist_ok=True)
os.makedirs("/tmp/models/latent_upscale_models", exist_ok=True)

print("✅ Workspace ready.")

### 📤 1. Restore the Custom Linker-Patched Binary Engine

Upload the `sd_cpp_cuda_built.tar.gz` archive from your local machine directly into the Kaggle environment (either via the **"Upload"** button in the Right Sidebar File Explorer under `/kaggle/working/`, or by mounting it as a custom private Dataset input). 

The following cell automatically detects where your archive is located, unpacks it directly into the execution path, and grants the necessary system execution privileges to the command-line interface.

In [ ]:
import os
import shutil

# Your exact copied path from the input dataset panel
dataset_bin_dir = "/kaggle/input/datasets/parijha/sdbuild/bin"

# Crucial: The destination MUST be our writeable /tmp partition execution path!
target_bin_dir = "/tmp/sd_bin/bin"

# Secure the target tree generation
os.makedirs(target_bin_dir, exist_ok=True)

if os.path.exists(dataset_bin_dir):
    print("📦 Unzipped build binaries discovered in your dataset mount!")
    print("🪄 Copying executable binaries to writeable workspace...")
    
    # Copy files cleanly from /kaggle/input into /tmp
    for item in os.listdir(dataset_bin_dir):
        source_item = os.path.join(dataset_bin_dir, item)
        target_item = os.path.join(target_bin_dir, item)
        if os.path.isfile(source_item):
            shutil.copy2(source_item, target_item)
            print(f"   ↳ Copied: {item} -> {target_bin_dir}")
            
    # Grant executable permissions to our newly copied binary inside /tmp
    print("\n🔐 Injecting runtime execution permissions...")
    !chmod +x /tmp/sd_bin/bin/sd-cli
    
    # Final health check confirmation to make sure it's where it belongs
    if os.path.exists('/tmp/sd_bin/bin/sd-cli'):
        print("\n🔥 SUCCESS: Engine fully restored, permissions set, and operational!")
    else:
        print("\n⚠️ Verification failed. Check if the /tmp system write limits were reached.")
else:
    print(f"❌ CRITICAL ERROR: Could not find binaries at {dataset_bin_dir}")
    print("Please verify the exact dataset link structure matches your workspace panel layout.")

### 📥 2. High-Speed Model Weights Acquisition

To ensure rapid downloads and maximize stability, this cell installs `aria2` (a high-speed multi-connection download utility). It then streams the hyper-optimized weights—including the **UD-IQ2_XXS** text encoder to minimize memory footprints and speed up generation—directly into the environment.

In [ ]:
# Install high-speed parallel downloader
print("📥 Installing aria2 framework...")
!apt-get update -qq && apt-get install -y -qq aria2

print("\n--- ⚡ Downloading LTX 2.3 Distilled Q3 Base Model (DiT) ---")
!aria2c -x 16 -s 16 -k 1M  -d "/tmp/models/diffusion_models" "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/distilled-1.1/ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf"

print("\n--- ⚡ Downloading Ultra-Optimized Gemma 3 12B UD-IQ2_XXS Text Encoder ---")
!aria2c -x 16 -s 16 -k 1M -d "/tmp/models/text_encoders" "https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/resolve/main/gemma-3-12b-it-UD-IQ2_XXS.gguf"

print("\n--- ⚡ Downloading Distilled Embeddings Connectors ---")
!aria2c -x 16 -s 16 -k 1M -d "/tmp/models/text_encoders" "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-distilled_embeddings_connectors.safetensors"

print("\n--- ⚡ Downloading Audio and Video Distilled VAEs ---")
!aria2c -x 16 -s 16 -k 1M -d "/tmp/models/vae" "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-distilled_video_vae.safetensors"
!aria2c -x 16 -s 16 -k 1M -d "/tmp/models/vae" "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-distilled_audio_vae.safetensors"

print("\n--- ⚡ Downloading LTX Spatial Latent Upscaler ---")
!aria2c -x 16 -s 16 -k 1M -d "/tmp/models/latent_upscale_models" "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"

print("\n📦 Download cycle completed.")

### 🧹 3. Raw Hash File-Mapping Correction

When pulling direct download endpoints from Hugging Face via `aria2c`, filenames can occasionally default to raw hexadecimal chunk hashes. The following script runs an internal directory check, audits the file configurations, and remaps them back to the explicit naming conventions required by the `stable-diffusion.cpp` engine.

In [ ]:
import os
import glob

print("🧹 Correcting model filename path structures...")

# 1. Main Base Model Mapping
dit_files = glob.glob("/tmp/models/diffusion_models/*")
if dit_files and not dit_files[0].endswith(".gguf"):
    os.rename(dit_files[0], "/tmp/models/diffusion_models/ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf")

# 2. Text Encoder & Connectors Sorting (Small file = connectors, Large file = Gemma)
te_files = sorted(glob.glob("/tmp/models/text_encoders/*"), key=os.path.getsize)
if len(te_files) >= 2:
    if not te_files[0].endswith(".safetensors"):
        os.rename(te_files[0], "/tmp/models/text_encoders/ltx-2.3-22b-distilled_embeddings_connectors.safetensors")
    if not te_files[1].endswith(".gguf"):
        os.rename(te_files[1], "/tmp/models/text_encoders/gemma-3-12b-it-UD-IQ2_XXS.gguf")

# 3. VAE Folder Sorting (Smaller file = audio VAE, Larger file = video VAE)
vae_files = sorted(glob.glob("/tmp/models/vae/*"), key=os.path.getsize)
if len(vae_files) >= 2:
    if not vae_files[0].endswith(".safetensors"):
        os.rename(vae_files[0], "/tmp/models/vae/ltx-2.3-22b-distilled_audio_vae.safetensors")
    if not vae_files[1].endswith(".safetensors"):
        os.rename(vae_files[1], "/tmp/models/vae/ltx-2.3-22b-distilled_video_vae.safetensors")

# 4. Latent Spatial Upscaler Correction
upscale_files = glob.glob("/tmp/models/latent_upscale_models/*")
if upscale_files and not upscale_files[0].endswith(".safetensors"):
    os.rename(upscale_files[0], "/tmp/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors")

print("\n🔍 Verifying clean directory tree structural health:")
!ls -R /tmp/models/

## Give permission to the SD executable.

In [ ]:
!chmod +x /tmp/sd_bin/bin/sd-server
import os
if os.access("/tmp/sd_bin/bin/sd-server", os.X_OK):
    print("✅ Permissions successfully set! Binary is now executable.")
else:
    print("❌ Failed to set permissions or binary not found.")

## 4a: Launch the Background Server

In [ ]:
import subprocess
import time
import os

# Set to False if you want to disable audio decoding and save time per generation.
LOAD_AUDIO_VAE = True
UPSCALER_MODEL = "/tmp/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
UPSCALER_DIR = os.path.dirname(UPSCALER_MODEL)

required_paths = [
    "/tmp/sd_bin/bin/sd-server",
    "/tmp/models/diffusion_models/ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf",
    "/tmp/models/vae/ltx-2.3-22b-distilled_video_vae.safetensors",
    "/tmp/models/text_encoders/gemma-3-12b-it-UD-IQ2_XXS.gguf",
    "/tmp/models/text_encoders/ltx-2.3-22b-distilled_embeddings_connectors.safetensors",
    UPSCALER_MODEL,
]
if LOAD_AUDIO_VAE:
    required_paths.append("/tmp/models/vae/ltx-2.3-22b-distilled_audio_vae.safetensors")

missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    raise FileNotFoundError("Missing required model/server files:\n" + "\n".join(missing_paths))

print("Starting stable-diffusion.cpp API server with LTX 2.3 model paths...")
print(f"Registering hires upscaler directory: {UPSCALER_DIR}")

server_cmd = [
    "/tmp/sd_bin/bin/sd-server",
    "--listen-ip", "127.0.0.1",
    "--listen-port", "1234",
    "--threads", "4",
    "--diffusion-model", "/tmp/models/diffusion_models/ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf",
    "--vae", "/tmp/models/vae/ltx-2.3-22b-distilled_video_vae.safetensors",
    "--llm", "/tmp/models/text_encoders/gemma-3-12b-it-UD-IQ2_XXS.gguf",
    "--embeddings-connectors", "/tmp/models/text_encoders/ltx-2.3-22b-distilled_embeddings_connectors.safetensors",
    "--hires-upscalers-dir", UPSCALER_DIR,
    "--diffusion-fa",
    "--offload-to-cpu",
    "--vae-tiling",
    "-v",
]

if LOAD_AUDIO_VAE:
    server_cmd += ["--audio-vae", "/tmp/models/vae/ltx-2.3-22b-distilled_audio_vae.safetensors"]

log_file = open("/kaggle/working/server.log", "w")
server_process = subprocess.Popen(server_cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("Waiting 90 seconds for models to load into RAM/VRAM...")
time.sleep(90)
print(f"Server check triggered. Mode: {'AUDIO+VIDEO' if LOAD_AUDIO_VAE else 'VIDEO ONLY'}")
print("If you changed this cell while an old server is running, stop/restart the Kaggle session before launching again.")


### 🏁 Verify the Engine Status

In [ ]:
!tail -n 20 /kaggle/working/server.log

## 💻 The Ultimate Beginner-Friendly Frontend Launch Cell

In [ ]:
import os
import glob
import time
import json
import base64
import requests
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*browser-compatible container.*")

try:
    import gradio as gr
except ImportError:
    print("Installing Gradio UI Framework...")
    !pip install -q gradio
    import gradio as gr

SERVER_URL = "http://127.0.0.1:1234"
UPSCALER_MODEL = "/tmp/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
UPSCALER_MODEL_NAME = os.path.splitext(os.path.basename(UPSCALER_MODEL))[0]

def get_vae_tiling_params(enable_upscale):
    if enable_upscale:
        return {
            "enabled": True,
            "temporal_tiling": True,
            "tile_size_x": 16,
            "tile_size_y": 16,
            "target_overlap": 0.25,
            "rel_size_x": 0.0,
            "rel_size_y": 0.0,
            "extra_tiling_args": "temporal_tile_frames=4,temporal_tile_overlap=1",
        }
    return {
        "enabled": True,
        "temporal_tiling": False,
        "tile_size_x": 32,
        "tile_size_y": 32,
        "target_overlap": 0.5,
        "rel_size_x": 0.0,
        "rel_size_y": 0.0,
        "extra_tiling_args": "",
    }

# -------------------------------------------------------------
# Pipeline interaction logic
# -------------------------------------------------------------
def get_live_logs():
    """Reads the tail end of the server log file to stream into the interface."""
    log_path = "/kaggle/working/server.log"
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            lines = f.readlines()
            return "".join(lines[-20:])
    return "Waiting for server logs to initialize..."

def scan_history():
    """Scans the working directory for generated video outputs."""
    video_files = glob.glob("/kaggle/working/gen_*.webm") + glob.glob("/kaggle/working/gen_*.avi")
    video_files.sort(key=os.path.getmtime, reverse=True)
    return video_files

def build_failure_message(status_res):
    """Turns the server job response and recent logs into a beginner-readable UI message."""
    parts = []
    for key in ("error", "message", "detail"):
        if status_res.get(key):
            parts.append(str(status_res[key]))

    result = status_res.get("result")
    if isinstance(result, dict):
        for key in ("error", "message", "detail"):
            if result.get(key):
                parts.append(str(result[key]))

    recent_logs = get_live_logs()
    important_logs = [
        line for line in recent_logs.splitlines()
        if "[ERROR]" in line or "[WARN" in line or "requires hires upscaler" in line.lower()
    ]
    if important_logs:
        parts.append("Recent engine log:\n" + "\n".join(important_logs[-6:]))
        lower_logs = "\n".join(important_logs).lower()
        if "out of memory" in lower_logs or "failed to allocate" in lower_logs:
            parts.append("What happened: the video was generated and upscaled, but final VAE decoding needed more GPU memory than available. Try the 360p preset, fewer frames, or keep the smaller upscale VAE tile settings in this notebook.")

    if not parts:
        parts.append(json.dumps(status_res, indent=2)[:2000])

    return "Video generation failed.\n\n" + "\n\n".join(parts)

def handle_generation(prompt, negative_prompt, steps, resolution_preset, use_custom_resolution, custom_width, custom_height, duration_seconds, input_image, enable_upscale):
    """Processes frontend inputs and posts generation parameters to the server."""
    if use_custom_resolution:
        width, height = int(custom_width), int(custom_height)
        if width % 32 != 0 or height % 32 != 0:
            raise gr.Error("Custom width and height must both be divisible by 32.")
        if width < 256 or height < 256:
            raise gr.Error("Custom width and height must be at least 256 pixels.")
        if width > 1920 or height > 1088:
            raise gr.Error("Custom resolution is capped at 1920x1088 for this Kaggle notebook.")
    elif "360p" in resolution_preset:
        width, height = 480, 360  # Proven fast baseline; the engine aligns height internally when needed.
    elif "480p" in resolution_preset:
        width, height = 640, 368  # Proven balanced size; the engine aligns height internally when needed.
    else:
        width, height = 832, 480

    fps = 12
    target_frames = max(9, int(round(float(duration_seconds) * fps)))
    frames = min(121, ((target_frames - 1 + 7) // 8) * 8 + 1)  # LTX video frame count rule: 8N + 1.

    payload = {
        "prompt": str(prompt),
        "negative_prompt": str(negative_prompt),
        "width": int(width),
        "height": int(height),
        "strength": 0.75 if input_image else 1.0,
        "seed": -1,
        "video_frames": int(frames),
        "fps": fps,
        "moe_boundary": 0.875,
        "vace_strength": 1.0,
        "sample_params": {
            "scheduler": "discrete",
            "sample_method": "euler",
            "sample_steps": int(steps),
            "flow_shift": 1.3568,
            "guidance": {"txt_cfg": 5.5, "img_cfg": 5.5, "distilled_guidance": 3.5},
        },
        "vae_tiling_params": get_vae_tiling_params(enable_upscale),
        "output_format": "avi",
        "output_compression": 100,
    }

    if "audio" not in payload["prompt"].lower():
        payload["prompt"] = f"{payload['prompt']}, high quality clear audio"

    if enable_upscale:
        if not os.path.exists(UPSCALER_MODEL):
            raise gr.Error(f"Upscaling is enabled, but the upscaler model is missing:\n{UPSCALER_MODEL}\n\nRun the download and filename correction cells first.")

        payload["hires"] = {
            "enabled": True,
            "upscaler": UPSCALER_MODEL_NAME,
            "scale": 2.0,
            "steps": 10,
            "denoising_strength": 0.7,
        }

    if input_image is not None and os.path.exists(input_image):
        with open(input_image, "rb") as img_file:
            img_base64 = base64.b64encode(img_file.read()).decode("utf-8")
        image_payload = f"data:image/png;base64,{img_base64}"
        payload["init_image"] = image_payload
        payload["input_image"] = image_payload

    try:
        r = requests.post(f"{SERVER_URL}/sdcpp/v1/vid_gen", json=payload, timeout=30)
        r.raise_for_status()
        job_id = r.json()["id"]

        while True:
            status_res = requests.get(f"{SERVER_URL}/sdcpp/v1/jobs/{job_id}", timeout=10).json()
            status = status_res.get("status", "unknown")

            if status == "completed":
                video_bytes = base64.b64decode(status_res["result"]["b64_json"])
                base_video_path = f"/kaggle/working/gen_{job_id}.avi"
                with open(base_video_path, "wb") as f:
                    f.write(video_bytes)
                return base_video_path

            if status in ("failed", "cancelled"):
                raise gr.Error(build_failure_message(status_res))

            time.sleep(4)

    except Exception as e:
        raise gr.Error(f"Could not communicate with the generation server.\n\n{type(e).__name__}: {e}\n\nRecent logs:\n{get_live_logs()}")

# -------------------------------------------------------------
# Visual interface
# -------------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# LTX-Video 2.3 Studio Cloud Interface")
    gr.Markdown("Generate videos with the controls below. Use the engine logs tab for live progress and error details.")

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Text Prompt", placeholder="Describe the video actions and sounds clearly...")
            neg_prompt = gr.Textbox(label="Negative Prompt", value="blurry, worst quality, low quality, glitch, distortion")

            resolution_preset = gr.Dropdown(
                choices=[
                    "360p (480x360) - Fastest Testing Baseline",
                    "480p (640x368) - Optimized Safe Balanced Size",
                    "720p (832x480) - High Resolution Cinematic Layout",
                ],
                value="480p (640x368) - Optimized Safe Balanced Size",
                label="Core Video Generation Dimensions",
            )

            use_custom_resolution = gr.Checkbox(label="Use Custom Resolution", value=False)
            with gr.Row():
                custom_width = gr.Slider(
                    minimum=256,
                    maximum=1920,
                    value=640,
                    step=32,
                    label="Custom Width",
                )
                custom_height = gr.Slider(
                    minimum=256,
                    maximum=1088,
                    value=384,
                    step=32,
                    label="Custom Height",
                )

            duration_seconds = gr.Slider(
                minimum=1,
                maximum=10,
                value=5,
                step=0.5,
                label="Duration Seconds (rounded to valid LTX frame count)",
            )

            steps = gr.Slider(
                minimum=4,
                maximum=30,
                value=8,
                step=1,
                label="Sampling Steps (LTX 2.3 Distilled Sweet Spot: 8-12)",
            )

            enable_upscale = gr.Checkbox(label="Enable Native Hi-Res Upscaling Pass", value=False)
            input_image = gr.Image(label="Input Image (For Image-to-Video)", type="filepath")
            generate_btn = gr.Button("Generate New Video", variant="primary")

        with gr.Column(scale=1):
            output_video = gr.Video(label="Generated Result Screen")
            with gr.Tab("Active Engine Logs"):
                log_box = gr.Textbox(label="Live C++ Terminal Output Stream", value="", lines=10, interactive=False)
                log_timer = gr.Timer(value=3.0, active=True)
                log_timer.tick(fn=get_live_logs, outputs=log_box)

            with gr.Tab("Generation History"):
                refresh_history_btn = gr.Button("Refresh History Archive", variant="secondary")
                history_gallery = gr.File(label="Generated Video Vault Files", file_count="multiple")

    generate_btn.click(
        fn=handle_generation,
        inputs=[prompt, neg_prompt, steps, resolution_preset, use_custom_resolution, custom_width, custom_height, duration_seconds, input_image, enable_upscale],
        outputs=output_video,
    ).then(fn=scan_history, outputs=history_gallery)

    refresh_history_btn.click(fn=scan_history, outputs=history_gallery)
    app.load(fn=scan_history, outputs=history_gallery)

app.queue()
app.launch(share=True, inline=False)


## Send Prompts via API (With Audio Mux Fix) SD Server NO UI (Optional)

In [ ]:
import base64
import time
import requests
import json
import os

base_url = "http://127.0.0.1:1234"
UPSCALER_MODEL = "/tmp/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
UPSCALER_MODEL_NAME = os.path.splitext(os.path.basename(UPSCALER_MODEL))[0]

payload = {
    "prompt": "a beautiful indian woman saying Hola, high quality clear feminine audio, synchronized voice sound effects, natural speech",
    "negative_prompt": "blurry, worst quality, low quality, glitch, distortion, silent, mute, no sound, static hiss",
    "width": 640,
    "height": 384,
    "strength": 0.75,
    "seed": 42,
    "video_frames": 65,
    "fps": 12,
    "moe_boundary": 0.875,
    "vace_strength": 1.0,
    "sample_params": {
        "scheduler": "discrete",
        "sample_method": "euler",
        "sample_steps": 20,
        "eta": 1.0,
        "shifted_timestep": 0,
        "custom_sigmas": [],
        "flow_shift": 1.3568,
        "guidance": {
            "txt_cfg": 5.5,
            "img_cfg": 5.5,
            "distilled_guidance": 3.5,
            "slg": {"layers": [7, 8, 9], "layer_start": 0.01, "layer_end": 0.2, "scale": 0.0},
        },
    },
    "high_noise_sample_params": {
        "scheduler": "discrete",
        "sample_method": "euler",
        "sample_steps": -1,
        "eta": 1.0,
        "shifted_timestep": 0,
        "flow_shift": 0.0,
        "guidance": {
            "txt_cfg": 7.0,
            "img_cfg": 7.0,
            "distilled_guidance": 3.5,
            "slg": {"layers": [7, 8, 9], "layer_start": 0.01, "layer_end": 0.2, "scale": 0.0},
        },
    },
    "hires": {
        "enabled": True,
        "upscaler": UPSCALER_MODEL_NAME,
        "scale": 2.0,
        "steps": 10,
        "denoising_strength": 0.7,
    },
    "lora": [],
    "vae_tiling_params": {
        "enabled": True,
        "temporal_tiling": True,
        "tile_size_x": 16,
        "tile_size_y": 16,
        "target_overlap": 0.25,
        "rel_size_x": 0.0,
        "rel_size_y": 0.0,
        "extra_tiling_args": "temporal_tile_frames=4,temporal_tile_overlap=1",
    },
    "cache_mode": "disabled",
    "cache_option": "",
    "scm_mask": "",
    "scm_policy_dynamic": True,
    "output_format": "avi",
    "output_compression": 100,
}

print("Submitting 'Hola' upscaled video job to server pipeline...")
try:
    if not os.path.exists(UPSCALER_MODEL):
        raise FileNotFoundError(f"Missing upscaler model: {UPSCALER_MODEL}")

    r = requests.post(f"{base_url}/sdcpp/v1/vid_gen", json=payload, timeout=30)
    r.raise_for_status()
    job_id = r.json()["id"]
    print(f"Job tracked successfully. ID: {job_id}\n")

    while True:
        status_res = requests.get(f"{base_url}/sdcpp/v1/jobs/{job_id}", timeout=10).json()
        status = status_res.get("status", "unknown")
        print(f"   State: {status.upper()}")

        if status in ("completed", "failed", "cancelled"):
            final_state = status_res
            break
        time.sleep(15)

    if final_state["status"] == "completed":
        print("\nServer generation completed. Decoding base64 video container...")
        video_bytes = base64.b64decode(final_state["result"]["b64_json"])
        output_filename = "/kaggle/working/output_woman_hola_upscaled.avi"
        with open(output_filename, "wb") as f:
            f.write(video_bytes)
        print(f"SUCCESS: Saved directly to: {output_filename}")
    else:
        print(f"\nFailure context: {json.dumps(final_state, indent=2)[:3000]}")

except Exception as e:
    print(f"\nScript Exception: {type(e).__name__}: {e}")


## 🎬 Hyper-Optimized Video Generation Target Execution Using SD CLI, Models will load each video generation. 

This cell runs your video prompt. It incorporates all of our system optimizations to prevent hardware bottlenecks on a single T4 GPU:
* **`--vae-tiling`**: Breaks down the huge 23.5 GB decode process into smaller chunks to prevent Out-Of-Memory errors.
* **`-W 640 -H 368`**: Balanced spatial aspect ratio to reduce processing overhead.
* **`--video-frames 65 --fps 12`**: Generates a smooth, 5-second structural sequence while cutting core processing time in half.

In [ ]:
# Define prompt variables for easy experimentation with sd-cli
PROMPT = "a cute Dog say hello"
NEGATIVE_PROMPT = "blurry, worst quality, low quality, glitch, distortion"

print(f"Launching generation sequence for prompt: '{PROMPT}'...")

!/tmp/sd_bin/bin/sd-cli -M vid_gen \
  --diffusion-model /tmp/models/diffusion_models/ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf \
  --vae /tmp/models/vae/ltx-2.3-22b-distilled_video_vae.safetensors \
  --audio-vae /tmp/models/vae/ltx-2.3-22b-distilled_audio_vae.safetensors \
  --llm /tmp/models/text_encoders/gemma-3-12b-it-UD-IQ2_XXS.gguf \
  --embeddings-connectors /tmp/models/text_encoders/ltx-2.3-22b-distilled_embeddings_connectors.safetensors \
  --hires-upscalers-dir /tmp/models/latent_upscale_models \
  -p "{PROMPT}" \
  -n "{NEGATIVE_PROMPT}" \
  --cfg-scale 5.5 \
  --sampling-method euler \
  -W 640 -H 384 \
  --diffusion-fa \
  --offload-to-cpu \
  --vae-tiling \
  --temporal-tiling \
  --vae-tile-size 16x16 \
  --vae-tile-overlap 0.25 \
  --extra-tiling-args "temporal_tile_frames=4,temporal_tile_overlap=1" \
  --video-frames 65 \
  --fps 12 \
  --hires \
  --hires-upscaler ltx-2.3-spatial-upscaler-x2-1.1 \
  --hires-scale 2.0 \
  --hires-steps 10 \
  --hires-denoising-strength 0.7 \
  -o /kaggle/working/output_video_optimized_upscaled.webm \
  -v
